# B2-022-probabilistic-latent-models — Practice p19 — Solution

**Type:** integrative · **Difficulty:** advanced · **Concepts:** variational-autoencoder

*65 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260927`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:F6-svd-spectral`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C9-dimensionality-reduction`, `book1:C11-neural-training`, `B2-021-cross-modal-transformers-vision`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:F6-svd-spectral](../../../../book1/units/F6-svd-spectral/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C9-dimensionality-reduction](../../../../book1/units/C9-dimensionality-reduction/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb).

## Supplied cell (copied verbatim from the statement)

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260927
LOADER_CANDIDATES = (
    Path("../data/latent_data.py"),
    Path("units/B2-022-probabilistic-latent-models/data/latent_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("latent_data", loader_path)
latent_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(latent_data)
torch.set_num_threads(1)

P12_X = torch.tensor([[1.0, 0.0, -1.0, 2.0], [0.5, 0.5, 0.5, 0.5], [-2.0, 1.0, 0.0, 1.0]])
P12_X_HAT = torch.tensor([[0.8, 0.1, -1.0, 1.5], [0.5, 0.0, 1.0, 0.5], [-1.0, 1.0, 0.5, 0.0]])
P12_MU = torch.tensor([[0.0, 0.0], [0.5, -0.5], [1.0, 2.0]])
P12_LOGVAR = torch.tensor([[0.0, 0.0], [-1.0, 0.0], [0.5, -0.5]])

## Solution

The loss is the p12 contract; the model is the pinned `TinyVAE` whose `forward(x, eps)` is deterministic given `eps`.
`train_vae` owns one `eps_generator` seeded with `SEED`, draws `eps` before each update, and records the batch-mean total, reconstruction, and KL from that update's forward pass (before the step).
A forward pre-hook verifies, during training only, that every row entering `forward` is a training row.

In [ ]:
import copy

DATASET = "mixture8d"


def train_rows():
    return latent_data.train_tensor(DATASET)


def heldout_rows():
    return latent_data.heldout_tensor(DATASET)


def split_hash_sets(name):
    hashes = latent_data.ROW_SHA256[name]
    train_part = {hashes[i] for i in latent_data.TRAIN_IDS[name]}
    heldout_part = {hashes[i] for i in latent_data.HELDOUT_IDS[name]}
    return train_part, heldout_part


TRAIN_HASHES, HELDOUT_HASHES = split_hash_sets(DATASET)


def install_training_row_guard(model):
    """Forward pre-hook: every row a training forward receives must hash into the training part."""
    seen = {"calls": 0, "rows": 0}

    def hook(module, args):
        batch = args[0]
        for row in batch:
            row_hash = latent_data.row_sha256(row)
            if row_hash not in TRAIN_HASHES or row_hash in HELDOUT_HASHES:
                raise AssertionError("a non-training row reached a training forward pass")
        seen["calls"] += 1
        seen["rows"] += batch.shape[0]

    return model.register_forward_pre_hook(hook), seen

def _check_inputs(x, x_hat, mu, logvar):
    for name, t in (("x", x), ("x_hat", x_hat), ("mu", mu), ("logvar", logvar)):
        if t.dim() != 2:
            raise ValueError(f"{name} must be 2-D")
    if x.shape != x_hat.shape:
        raise ValueError("x and x_hat shapes differ")
    if mu.shape != logvar.shape:
        raise ValueError("mu and logvar shapes differ")
    if x.shape[0] != mu.shape[0]:
        raise ValueError("batch sizes differ")


def negative_elbo_terms(x, x_hat, mu, logvar):
    _check_inputs(x, x_hat, mu, logvar)
    recon = 0.5 * ((x - x_hat) ** 2).sum(dim=1)                       # sum over features
    kl = 0.5 * (mu ** 2 + logvar.exp() - logvar - 1).sum(dim=1)       # sum over latents
    return recon, kl


def negative_elbo(x, x_hat, mu, logvar):
    recon, kl = negative_elbo_terms(x, x_hat, mu, logvar)
    return (recon + kl).mean()                                         # mean over the batch

class TinyVAE(nn.Module):
    def __init__(self, in_dim=8, hidden=16, latent=2):
        super().__init__()
        self.hidden = nn.Linear(in_dim, hidden)
        self.mu_head = nn.Linear(hidden, latent)
        self.logvar_head = nn.Linear(hidden, latent)
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, in_dim))

    def encode(self, x):
        h = F.relu(self.hidden(x))
        return self.mu_head(h), self.logvar_head(h)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, eps):
        mu, logvar = self.encode(x)
        z = mu + torch.exp(0.5 * logvar) * eps
        return self.decode(z), mu, logvar


def train_vae(model, batch, optimizer):
    eps_generator = torch.Generator().manual_seed(SEED)
    trace = []
    for _ in range(600):
        eps = torch.randn(batch.shape[0], 2, generator=eps_generator)
        optimizer.zero_grad(set_to_none=True)
        x_hat, mu, logvar = model(batch, eps)
        recon, kl = negative_elbo_terms(batch, x_hat, mu, logvar)
        loss = negative_elbo(batch, x_hat, mu, logvar)   # the pinned training loss
        trace.append({"total": loss.item(), "recon": recon.mean().item(), "kl": kl.mean().item()})
        loss.backward()
        optimizer.step()
    return trace


def heldout_eval(model, eval_eps):
    H = heldout_rows()
    with torch.no_grad():
        x_hat, mu, logvar = model(H, eval_eps)
        recon, kl = negative_elbo_terms(H, x_hat, mu, logvar)
        return {"total": negative_elbo(H, x_hat, mu, logvar).item(), "recon": recon.mean().item(),
                "kl": kl.mean().item(), "shapes": (tuple(x_hat.shape), tuple(mu.shape), tuple(logvar.shape))}


p12_value = negative_elbo(P12_X, P12_X_HAT, P12_MU, P12_LOGVAR)

torch.manual_seed(SEED)
model = TinyVAE()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0)
eval_eps = torch.randn(12, 2, generator=torch.Generator().manual_seed(SEED + 1))

heldout_before = heldout_eval(model, eval_eps)
params_before = copy.deepcopy({name: p.detach() for name, p in model.named_parameters()})
guard, guard_stats = install_training_row_guard(model)
trace = train_vae(model, train_rows(), optimizer)
guard.remove()
heldout_after = heldout_eval(model, eval_eps)
changed = [name for name, p in model.named_parameters() if not torch.equal(p.detach(), params_before[name])]
last20 = sum(r["total"] for r in trace[-20:]) / 20
print("first", trace[0], "last-20 mean total", round(last20, 4))
print("held-out before", heldout_before)
print("held-out after ", heldout_after)

### Interpretation

After training the held-out reconstruction term is about 1.06 nats per row (half the summed squared error over 8 features, down from about 8.3 before training), so held-out rows are reconstructed far better than at initialization from only a 2-D noisy code.
The held-out mean KL is about 1.47 nats, well above `0.1`, which means the encoder's posteriors differ substantially from the prior across rows — the latent carries information instead of collapsing to $N(0,I)$.
The ELBO trades these off: the model pays about 1.5 nats of KL to buy roughly 7 nats of reconstruction improvement, and since `total = recon + kl` holds exactly for each record the bookkeeping is consistent.
Each update uses fresh `eps`, so single-update totals are noisy Monte Carlo estimates, and averaging the last 20 updates smooths that sampling noise so the decrease certificate is not decided by one lucky or unlucky draw.

### Answer check

In [ ]:
assert torch.allclose(p12_value, torch.tensor(1.5288552), atol=1e-6, rtol=1e-6)
assert train_rows().shape == (48, 8) and heldout_rows().shape == (12, 8)
assert train_rows().dtype == torch.float32 and heldout_rows().dtype == torch.float32
assert heldout_after["shapes"] == ((12, 8), (12, 2), (12, 2))
assert len(trace) == 600
for record in trace:
    assert abs(record["total"] - (record["recon"] + record["kl"])) <= 1e-5
    assert record["kl"] >= 0
assert last20 <= 0.6 * trace[0]["total"]
assert heldout_after["total"] <= 0.6 * heldout_before["total"]
assert heldout_after["kl"] >= 0.1
assert len(changed) >= 1
assert guard_stats == {"calls": 600, "rows": 600 * 48}
assert all(latent_data.row_sha256(row) in TRAIN_HASHES for row in train_rows())
assert not any(latent_data.row_sha256(row) in HELDOUT_HASHES for row in train_rows())